Preprocessing

In [2]:
import pandas as pd
from datetime import datetime
from methods import *

df = pd.read_csv("data/archive/master.csv")
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
# rows only useful if the fight ended up with somebody winning (and not something like a no contest)
df = df[df["result_status"] == "win"]

# dropping stances for now for simplicities' sake
# will encode at a later date
df.drop(columns=["r_stance", "b_stance"], inplace=True)

# df = pd.get_dummies(data=df, columns=['r_stance'], prefix='r_stance')
# df = pd.get_dummies(data=df, columns=['b_stance'], prefix='b_stance')
# df.head(5)

# convert feet and inches to cms for height columns
df["r_height"] = df["r_height"].apply(convert_to_cms)
df["b_height"] = df["b_height"].apply(convert_to_cms)

# pair fighter names to their ids for easy look up
fighter_name_to_id = {}
for idx, row in df.reset_index(drop=True).iterrows():
    r_fighter_name = row["r_fighter_name"]
    b_fighter_name = row["b_fighter_name"]
    if r_fighter_name not in fighter_name_to_id:
        fighter_name_to_id[r_fighter_name] = row["r_fighter_id"]
    if b_fighter_name not in fighter_name_to_id:
        fighter_name_to_id[b_fighter_name] = row["b_fighter_id"]

# sort dataframe by event date so it's in chronological order
df = df.sort_values(by='event_date')
# df
fighter_name_to_id

C:\Users\samro\AppData\Local\Temp\ipykernel_15200\950586975.py:12: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["r_stance", "b_stance"], inplace=True)


{'Pete Spratt': 'c3ea8deeba00462b',
 'Jeremy Jackson': 'ad32471f01e7b1a5',
 'Nick Diaz': 'f57a8a52ca401ed9',
 'Joe Stevenson': '79899ecf62020f6d',
 'Jerry Bohlander': '93892752c5fc23ca',
 'Kevin Jackson': '13b2f59210dda9cc',
 'Frank Shamrock': 'fcaae0385b514f11',
 'Tony Fryklund': '253de74457bd6374',
 'Todd Butler': '3fed746acfd026dd',
 'Dean Barry': 'a6f3d15cb84334bb',
 'Mike Jackson': '0641c21bec209472',
 'Pete Rodriguez': '776d4262c4da38bf',
 'Brad Pickett': 'c945adc22c2bfe8f',
 'Yves Jabouin': 'b4eeac608fc82f88',
 'Mickey Gall': '9e50097a89442158',
 'Walel Watson': '92aafa8f4b26269b',
 'Eugene Jackson': 'ce783bf73b5131f9',
 'Mike Seal': '0c01568b1ac77bff',
 'Ian Loveland': 'eea0274e153256a0',
 'Felipe Arantes': '71002dbc1743b9a7',
 'Joe Riggs': 'd512d9f204059f57',
 'Pablo Garza': '8e5bfd4e82352340',
 'Eddie Wineland': 'fc9a9559a05f2704',
 'Mike Easton': 'bc7230f231701d66',
 'Thomas Almeida': '2b074403b7c6cdb4',
 'Jeff Hougland': '97a8f39186f489db',
 'Raphael Assuncao': '2f13e4020ce

From the master dataframe, build a new dataframe that contains the running accumulation of stats for both fighters up until the fight

In [3]:
result = build_accum_stats_df(df)
final_df = result[0]
accum_stats_dict = result[1]
# final_df
accum_stats_dict
    

{'598a58db87b890ee': {'sum_kd': 2,
  'sum_sig_landed': 15,
  'sum_sig_atmp': 30,
  'sum_td_success': 0,
  'sum_td_atmp': 0,
  'sum_sub_att': 0,
  'sum_rev': 0,
  'sum_sig_str_landed_head': 12,
  'sum_sig_str_atmp_head': 24,
  'sum_sig_str_landed_body': 3,
  'sum_sig_str_atmp_body': 5,
  'sum_sig_str_landed_leg': 0,
  'sum_sig_str_atmp_leg': 1,
  'sum_sig_str_landed_distance': 4,
  'sum_sig_str_atmp_distance': 13,
  'sum_sig_str_landed_clinch': 4,
  'sum_sig_str_atmp_clinch': 9,
  'sum_sig_str_landed_ground': 7,
  'sum_sig_str_atmp_ground': 8,
  'sum_ctrl_seconds': 0,
  'height': 193.04,
  'weight_lbs': 275.0,
  'reach_inches': nan,
  'sum_rounds_fought': 2},
 'd3711d3784b76255': {'sum_kd': 0,
  'sum_sig_landed': 13,
  'sum_sig_atmp': 30,
  'sum_td_success': 0,
  'sum_td_atmp': 0,
  'sum_sub_att': 0,
  'sum_rev': 0,
  'sum_sig_str_landed_head': 7,
  'sum_sig_str_atmp_head': 20,
  'sum_sig_str_landed_body': 4,
  'sum_sig_str_atmp_body': 7,
  'sum_sig_str_landed_leg': 2,
  'sum_sig_str_at

Oversampling to fix imbalanced data, encoding target values

In [6]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = final_df.iloc[:, 0:48]
y = final_df.iloc[:, 48]

ros = RandomOverSampler(sampling_strategy="not majority")
X_res, y_res = ros.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, random_state=17, test_size=0.2)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=17)

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

# to check what labels were given what code
res = {}
for cl in le.classes_:
    res.update({cl: le.transform([cl])})
res

{'b': array([0]), 'r': array([1])}

Decision Tree

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

dtc = DecisionTreeClassifier(random_state=17)
dtc.fit(X_train, y_train)
y_pred = dtc.predict(X_val)

val_accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"\nClassification Resport:\n", classification_report(y_val, y_pred))
print(f"\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))

Validation Accuracy: 0.7597

Classification Resport:
               precision    recall  f1-score   support

           0       0.73      0.82      0.77      1516
           1       0.80      0.70      0.75      1551

    accuracy                           0.76      3067
   macro avg       0.76      0.76      0.76      3067
weighted avg       0.76      0.76      0.76      3067


Confusion Matrix:
 [[1244  272]
 [ 465 1086]]


Random Forest

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=17)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)

val_accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"\nClassification Resport:\n", classification_report(y_val, y_pred))
print(f"\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))

Validation Accuracy: 0.7760

Classification Resport:
               precision    recall  f1-score   support

           0       0.74      0.85      0.79      1516
           1       0.82      0.71      0.76      1551

    accuracy                           0.78      3067
   macro avg       0.78      0.78      0.78      3067
weighted avg       0.78      0.78      0.78      3067


Confusion Matrix:
 [[1283  233]
 [ 454 1097]]


XGBoost

In [ ]:
import xgboost as xgb

xgb_classifier = xgb.XGBClassifier(random_state=17)
xgb_classifier.fit(X_train, y_train)
y_pred = xgb_classifier.predict(X_val)

val_accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"\nClassification Resport:\n", classification_report(y_val, y_pred))
print(f"\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))


Hyperparameter Tuning Random Forest

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [10, 100, 200, 300],
    "max_depth": [3, 5, 10, 20, None],
    "max_features": ["sqrt", "log2", None],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

rf = RandomForestClassifier(random_state=17)
grid_search = GridSearchCV(rf, param_grid, cv=10, scoring="accuracy", n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_

print(f"Best parameters: ", grid_search.best_params_)
print(f"Best score: ", grid_search.best_score_)

In [ ]:
y_pred = best_rf.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"\nClassification Resport:\n", classification_report(y_test, y_pred))
print(f"\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

XGBoost Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators": [100, 200, 300, 400, 500],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 6, 9],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1, 10, 100],
    "reg_lambda": [0.5, 0.7, 1, 1.3]
}

xgb_classifier = xgb.XGBClassifier(random_state=17)

rand_search = RandomizedSearchCV(xgb_classifier, param_grid, cv=10, scoring="accuracy", n_iter=100, n_jobs=-1, verbose=2)
rand_search.fit(X_train, y_train)

best_xgb = rand_search.best_estimator_

print(f"Best parameters: ", rand_search.best_params_)
print(f"Best score: ", rand_search.best_score_)

In [ ]:
y_pred = best_xgb.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"\nClassification Resport:\n", classification_report(y_test, y_pred))
print(f"\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Predicting UFC 333 on 24/10/2026

In [12]:
print("RANDOM FOREST PREDICTIONS (NO HYPERTUNING)")
print("*" * 50)
print(predict_fight(["Movsar Evloev", "Alexander Volkanovski"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Merab Dvalishvili", "Petr Yan"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Lone'er Kavanagh", "Ramazan Temirov"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Alexander Volkov", "Rizvan Kuniev"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Aaron Pico", "Losene Keita"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Azamat Murzakanov", "Dominick Reyes"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Nikita Krylov", "Abdul Rakhman Yakhyaev"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Abus Magomedov", "Cam Rowston"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Grant Dawson", "Nurullo Aliev"], fighter_name_to_id, accum_stats_dict, rf))
# print(predict_fight(["Colby Thicknesse", "Asaf Chopurov"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Abubakar Vagaev", "Jacob Malkoun"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Bolaji Oki", "Daniel Santos"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Kody Steele", "Noah Gugnon"], fighter_name_to_id, accum_stats_dict, rf))
print(predict_fight(["Iasmin Lucindo", "Nicolle Caliari"], fighter_name_to_id, accum_stats_dict, rf))

# print("\nXGBOOST PREDICTIONS (NO HYPERTUNING)")
# print("*" * 50)
# print(predict_fight(["Movsar Evloev", "Alexander Volkanovski"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# print(predict_fight(["Merab Dvalishvili", "Petr Yan"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# print(predict_fight(["Lone'er Kavanagh", "Ramazan Temirov"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# print(predict_fight(["Alexander Volkov", "Rizvan Kuniev"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# print(predict_fight(["Aaron Pico", "Losene Keita"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# print(predict_fight(["Azamat Murzakanov", "Dominick Reyes"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# print(predict_fight(["Nikita Krylov", "Abdul Rakhman Yakhyaev"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# print(predict_fight(["Abus Magomedov", "Cam Rowston"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# print(predict_fight(["Grant Dawson", "Nurullo Aliev"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# # print(predict_fight(["Colby Thicknesse", "Asaf Chopurov"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# print(predict_fight(["Abubakar Vagaev", "Jacob Malkoun"], fighter_name_to_id, accum_stats_dict, rf))
# print(predict_fight(["Bolaji Oki", "Daniel Santos"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# print(predict_fight(["Kody Steele", "Noah Gugnon"], fighter_name_to_id, accum_stats_dict, xgb_classifier))
# print(predict_fight(["Iasmin Lucindo", "Nicolle Caliari"], fighter_name_to_id, accum_stats_dict, xgb_classifier))

# print("RANDOM FOREST PREDICTIONS (WITH HYPERTUNING)")
# print("*" * 50)
# print(predict_fight(["Movsar Evloev", "Alexander Volkanovski"], fighter_name_to_id, accum_stats_dict, best_rf))
# print(predict_fight(["Merab Dvalishvili", "Petr Yan"], fighter_name_to_id, accum_stats_dict, best_rf))
# print(predict_fight(["Lone'er Kavanagh", "Ramazan Temirov"], fighter_name_to_id, accum_stats_dict, best_rf))
# print(predict_fight(["Alexander Volkov", "Rizvan Kuniev"], fighter_name_to_id, accum_stats_dict, best_rf))
# print(predict_fight(["Aaron Pico", "Losene Keita"], fighter_name_to_id, accum_stats_dict, best_rf))
# print(predict_fight(["Azamat Murzakanov", "Dominick Reyes"], fighter_name_to_id, accum_stats_dict, best_rf))
# print(predict_fight(["Nikita Krylov", "Abdul Rakhman Yakhyaev"], fighter_name_to_id, accum_stats_dict, best_rf))
# print(predict_fight(["Abus Magomedov", "Cam Rowston"], fighter_name_to_id, accum_stats_dict, best_rf))
# print(predict_fight(["Grant Dawson", "Nurullo Aliev"], fighter_name_to_id, accum_stats_dict, best_rf))
# # print(predict_fight(["Colby Thicknesse", "Asaf Chopurov"], fighter_name_to_id, accum_stats_dict, best_rf))
# print(predict_fight(["Bolaji Oki", "Daniel Santos"], fighter_name_to_id, accum_stats_dict, best_rf))
# print(predict_fight(["Kody Steele", "Noah Gugnon"], fighter_name_to_id, accum_stats_dict, best_rf))
# print(predict_fight(["Iasmin Lucindo", "Nicolle Caliari"], fighter_name_to_id, accum_stats_dict, best_rf))

# print("XGBOOST PREDICTIONS (WITH HYPERTUNING)")
# print("*" * 50)
# print(predict_fight(["Movsar Evloev", "Alexander Volkanovski"], fighter_name_to_id, accum_stats_dict, best_xgb))
# print(predict_fight(["Merab Dvalishvili", "Petr Yan"], fighter_name_to_id, accum_stats_dict, best_xgb))
# print(predict_fight(["Lone'er Kavanagh", "Ramazan Temirov"], fighter_name_to_id, accum_stats_dict, best_xgb))
# print(predict_fight(["Alexander Volkov", "Rizvan Kuniev"], fighter_name_to_id, accum_stats_dict, best_xgb))
# print(predict_fight(["Aaron Pico", "Losene Keita"], fighter_name_to_id, accum_stats_dict, best_xgb))
# print(predict_fight(["Azamat Murzakanov", "Dominick Reyes"], fighter_name_to_id, accum_stats_dict, best_xgb))
# print(predict_fight(["Nikita Krylov", "Abdul Rakhman Yakhyaev"], fighter_name_to_id, accum_stats_dict, best_xgb))
# print(predict_fight(["Abus Magomedov", "Cam Rowston"], fighter_name_to_id, accum_stats_dict, best_xgb))
# print(predict_fight(["Grant Dawson", "Nurullo Aliev"], fighter_name_to_id, accum_stats_dict, best_xgb))
# # print(predict_fight(["Colby Thicknesse", "Asaf Chopurov"], fighter_name_to_id, accum_stats_dict, best_xgb))
# print(predict_fight(["Bolaji Oki", "Daniel Santos"], fighter_name_to_id, accum_stats_dict, best_xgb))
# print(predict_fight(["Kody Steele", "Noah Gugnon"], fighter_name_to_id, accum_stats_dict, best_xgb))
# print(predict_fight(["Iasmin Lucindo", "Nicolle Caliari"], fighter_name_to_id, accum_stats_dict, best_xgb))

RANDOM FOREST PREDICTIONS (NO HYPERTUNING)
**************************************************
[[0.44 0.56]]
[[0.71 0.29]]
[[0.54 0.46]]
[[0.6 0.4]]
[[0.45 0.55]]
[[0.51 0.49]]
[[0.64 0.36]]
[[0.58 0.42]]
[[0.61 0.39]]
[[0.35 0.65]]
[[0.62 0.38]]
[[0.39 0.61]]
[[0.65 0.35]]


Save Random Forest Model

In [11]:
import joblib

joblib.dump(rf, "./models/random_forest_model.joblib")
joblib.dump(fighter_name_to_id, "./fighters_data/fighter_name_to_id.joblib")
joblib.dump(accum_stats_dict, "./fighters_data/fighters_accum_stats.joblib")

['./fighters_data/fighters_accum_stats.joblib']

In [10]:
print(predict_fight(["Randy Brown", "Jacobe Smith"], fighter_name_to_id, accum_stats_dict, rf))

[[0.83 0.17]]
